# 🍥 AI Dubbing Studio — Kokoro-82M TTS + RVC (RMVPE)
### Optimized for Google Colab Free T4 GPU

This notebook runs the complete end-to-end multi-language anime/video dubbing pipeline:
- 🧠 **Kokoro-82M Neural TTS:** 100% free, local text-to-speech with native Hindi, Spanish, French, and Portuguese voices.
- 🎙️ **RVC Voice Conversion (RMVPE):** AI voice cloning with RMVPE pitch extraction accelerated on T4 GPU.
- ⚡ **CUDA & ONNX-GPU:** Explicit hardware acceleration targeting `cuda:0` and `onnxruntime-gpu`.
- 📂 **Persistent Colab Storage:** Saves all masters directly to `/content/outputs/` for easy download.
- 🌐 **Gradio Live UI:** Public shareable URL (`.gradio.live`) streaming live progress without timeouts.

👉 **Requirement:** Go to `Runtime` -> `Change runtime type` -> select **T4 GPU** before running.

In [ ]:
# ============================================================
# STEP 1: Verify Active GPU Acceleration
# ============================================================
!nvidia-smi

In [ ]:
# ============================================================
# STEP 2: Install ALL Dependencies (Zero-Hang, T4 GPU Matrix)
# ============================================================
# ─── 2a. System packages ────────────────────────────────────
!apt-get update -qq && apt-get install -y -qq ffmpeg espeak-ng build-essential

# ─── 2b. Clone / update project ─────────────────────────────
!git clone https://github.com/adarshraj990/-narutogen.git /content/narutogen 2>/dev/null || \
    (cd /content/narutogen && git pull --rebase origin main)
%cd /content/narutogen

# ─── 2c. Pin NumPy < 2.0 FIRST (prevents ABI crashes across all packages) ───
!pip install -q "numpy<2.0.0" cython setuptools wheel

# ─── 2d. Install pyworld from the pre-built wheel (skips ALL C++ compilation) ───
!pip install -q pyworld-prebuilt

# ─── 2e. Install fairseq from binary ONLY — no source compilation ───────────
# fairseq-fixed provides a pre-built wheel; if it fails, skip it because
# rvc_python bundles a compatible fairseq copy internally.
!pip install -q --prefer-binary fairseq-fixed || echo '[INFO] fairseq-fixed binary not found; rvc-python will use its bundled copy.'

# ─── 2f. Install rvc-python with NO GPU kernel compilation ──────────────────
# NO_CUDA=1 prevents rvc-python from compiling CUDA C++ extensions at install-time
# (those extensions are compiled lazily at first inference on the real GPU instead).
!NO_CUDA=1 pip install -q --prefer-binary rvc-python

# ─── 2g. GPU-accelerated FAISS + ONNX Runtime (T4 CUDA 12) ─────────────────
!pip install -q faiss-gpu-cu12 onnxruntime-gpu || pip install -q faiss-cpu onnxruntime-gpu

# ─── 2h. All remaining runtime deps ─────────────────────────────────────────
!pip install -q \
    hydra-core \
    omegaconf \
    "antlr4-python3-runtime==4.9.3" \
    kokoro \
    soundfile \
    pydub \
    pysrt \
    "gradio>=4.44.1" \
    librosa \
    huggingface_hub \
    requests \
    psutil

# ─── 2i. Verify import (loudly fails if rvc-python isn't importable) ────────
import subprocess, sys
result = subprocess.run(
    [sys.executable, '-c', 'from rvc_python.infer import RVCInference; print("✅ rvc_python OK:", RVCInference)'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)
if result.returncode != 0:
    raise RuntimeError('🚨 rvc_python failed to import! Check the output above for the real error.')

print('\n✅ All Colab T4 GPU AI dependencies installed and verified successfully!')

In [ ]:
# ============================================================
# STEP 3: Launch AI Dubbing Studio
# ============================================================
# - Generates a public https://xxxx.gradio.live link
# - Streams ALL pipeline logs (including RVC errors) to cell output
# - Saves master audio files to /content/outputs/
#
# ⚠️  If Step 2's rvc_python verification passed but you see an
#      import error here, click Runtime → Restart runtime, then
#      skip Step 2 and run only Step 3.
!python app.py